# Week 8 - Deep Learning for Representations and Dynamics: from PCA to Autoencoders to SHRED

Deep learning can feel like a discontinuity in the course -- as though everything
we built with the singular value decomposition were suddenly replaced by an
opaque network of weights. This lesson makes the opposite case, then shows where
the networks genuinely go beyond the SVD. The simplest neural network that learns
a representation, a **linear autoencoder** trained to minimize reconstruction
error, is not a rival to PCA: it *is* PCA. The Eckart-Young theorem guarantees
that the best rank-$k$ approximation of a matrix is its truncated SVD, so gradient
descent walks a linear autoencoder straight to the principal subspace we already
know how to compute in closed form.

That equivalence is the *floor*. The rest of the lesson relaxes the "linear"
assumption and measures what it buys. A **nonlinear autoencoder** bends its latent
surface to follow a curved manifold that no flat plane can capture; and **SHRED**
(a shallow recurrent decoder) reconstructs an entire spatiotemporal field from a
handful of sensors -- the complement to Week 7's question of how far ahead you can
forecast. We train these with **PyTorch**, device-agnostic: the code uses a GPU
when one is available (as on Colab) and falls back to the CPU otherwise, and the
models are deliberately small so the whole lesson runs in seconds either way.

The genuinely large models -- transformers and foundation-model embeddings for
sequences and clinical text, and reinforcement learning for sequential decisions
such as dosing -- are the subject of the **capstone project**, where they are
taught through a real case study.

**Reading.** Kutz, *Data-Driven Modeling & Scientific Computation* / *Data-Driven
Science and Engineering*, Chapters 15 (neural networks and deep learning) and 19
(SHRED and sensor-based reconstruction). Read those chapters for the architectures
and their derivations; everything below is developed in our own terms and
validated against our own fixtures.

**Learning goals.**

- State and *demonstrate numerically* the equivalence between a linear
  autoencoder trained by gradient descent and PCA / the truncated SVD.
- Read a training-loss curve and confirm it descends to the closed-form
  Eckart-Young optimum rather than beating it.
- Apply the same linear-representation machinery to a real biomedical feature
  matrix and interpret the resulting two-dimensional latent space.
- Train a **nonlinear autoencoder** in PyTorch and show it recovers a curved
  manifold that the best linear latent cannot.
- Use **SHRED** to reconstruct a full spatiotemporal field from a few sensors,
  and read how the reconstruction improves as sensors are added.
- Close with an explicit, calibrated confidence statement -- honest that the
  nonlinear gains, unlike the linear=PCA equivalence, carry no closed-form
  guarantee.

## Setup

We seed every random number generator and apply the course plotting style so the
figures below are deterministic and reproducible from a cold kernel, and we select
a compute device (GPU if available, otherwise CPU).

In [ ]:
# Colab setup: install the ddm4bio course library.
# No-op when ddm4bio is already importable (e.g. the course-site build), so
# this cell is safe everywhere. It is hidden from the rendered site via the
# "remove-cell" tag, but runs when this notebook is opened in Google Colab.
try:
    import ddm4bio  # noqa: F401
except ModuleNotFoundError:
    %pip install -q "ddm4bio @ git+https://github.com/symbiont-ai/ddm4bio.git"
    import ddm4bio  # noqa: F401

In [ ]:
import numpy as np
import torch

import ddm4bio
from ddm4bio import seed_everything
from ddm4bio.viz.style import set_style

seed_everything()
torch.manual_seed(0)
set_style()

# Device-agnostic: use a GPU when one is available (e.g. Colab), else the CPU.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"ddm4bio version: {ddm4bio.__version__}")
print(f"PyTorch {torch.__version__} on device: {device}")

## 1. The equivalence, on synthetic ground truth

A **linear autoencoder** is the smallest network that learns a representation. It
compresses each centered input row $x \in \mathbb{R}^d$ to a $k$-dimensional code
$z = x W_e$ with an encoder matrix $W_e \in \mathbb{R}^{d \times k}$, then expands
it back with a decoder matrix $W_d \in \mathbb{R}^{k \times d}$ to a
reconstruction $\hat{x} = z W_d$. There are no nonlinear activations anywhere --
that omission is the whole point. Training minimizes the mean squared
reconstruction error $\frac{1}{N d}\sum_i \lVert \hat{x}_i - x_i \rVert^2$ by
gradient descent on the two weight matrices.

Here is the claim we will verify. Among *all* rank-$k$ linear maps, the
Eckart-Young theorem says the best squared-error reconstruction of a centered
data matrix is its truncated SVD -- exactly what PCA computes. A linear
autoencoder can only ever realize a rank-$k$ linear map, so the lowest loss it can
possibly reach is the PCA reconstruction error. Gradient descent, given enough
steps, converges to that floor. The encoder/decoder it lands on spans the **same
subspace** as the top-$k$ principal components (up to a rotation and scaling
inside the code, which the reconstruction is blind to).

To test this cleanly we build data whose true rank we control: 300 samples living
on a random 3-dimensional subspace inside 12-dimensional feature space, plus a thin
layer of isotropic noise. An honest rank-3 reconstruction should capture almost
everything.

In [ ]:
from ddm4bio.methods.decomposition import explained_variance_ratio, svd_lowrank
from ddm4bio.methods.validation import reconstruction_error

rng = np.random.default_rng(0)

n_samples, n_features, true_rank = 300, 12, 3

# Rank-3 generator: latent scores (300 x 3) times loadings (3 x 12), plus noise.
latent = rng.standard_normal((n_samples, true_rank))
loadings = rng.standard_normal((true_rank, n_features))
noise = 0.02 * rng.standard_normal((n_samples, n_features))
X = latent @ loadings + noise

# Center once; both PCA and the autoencoder operate on the centered matrix.
Xc = X - X.mean(axis=0, keepdims=True)

evr = explained_variance_ratio(X)
print(f"Data matrix X: {X.shape} (samples x features)")
print(f"Explained-variance ratio (first 6 components):\n{np.round(evr[:6], 4)}")
print(f"First {true_rank} components capture {evr[:true_rank].sum():.4%} of variance.")

### 1a. The PCA reconstruction (closed form)

The rank-$k$ PCA reconstruction is the truncated SVD of the centered matrix:
keep the top $k$ singular triplets and multiply them back together. This is the
Eckart-Young optimum -- no rank-$k$ linear map reconstructs `Xc` with lower
squared error.

In [ ]:
k = true_rank

U, s, Vt = svd_lowrank(Xc, k)
X_pca = U @ np.diag(s) @ Vt  # best rank-k reconstruction (Eckart-Young)

err_pca = reconstruction_error(Xc, X_pca, kind="rel_l2")
print(f"PCA (rank-{k}) relative-L2 reconstruction error: {err_pca:.6f}")

### 1b. The linear autoencoder (gradient descent)

Now the network. We initialize small random encoder and decoder matrices and run
plain full-batch gradient descent on the mean-squared reconstruction error. The
gradients are elementary -- no autodiff framework, no GPU, just two matrix
multiplies per step -- which is exactly why this fits in an offline notebook.

In [ ]:
d = n_features
N = Xc.shape[0]

# Small random initial weights (a fresh, seeded generator for reproducibility).
init_rng = np.random.default_rng(1)
W_e = 0.1 * init_rng.standard_normal((d, k))  # encoder: features -> code
W_d = 0.1 * init_rng.standard_normal((k, d))  # decoder: code -> features

lr = 0.05
n_epochs = 4000
losses = np.empty(n_epochs)

for epoch in range(n_epochs):
    Z = Xc @ W_e            # (N, k) latent codes
    X_hat = Z @ W_d         # (N, d) reconstruction
    R = X_hat - Xc          # residual
    losses[epoch] = np.mean(R**2)

    # Gradients of the mean-squared error w.r.t. each weight matrix.
    grad_W_d = (Z.T @ R) * (2.0 / (N * d))
    grad_W_e = (Xc.T @ (R @ W_d.T)) * (2.0 / (N * d))

    W_e -= lr * grad_W_e
    W_d -= lr * grad_W_d

X_ae = (Xc @ W_e) @ W_d
err_ae = reconstruction_error(Xc, X_ae, kind="rel_l2")
print(f"Autoencoder (rank-{k}) relative-L2 reconstruction error: {err_ae:.6f}")
print(f"Final training MSE: {losses[-1]:.3e}")

The training loss should fall and then flatten onto the PCA reconstruction error.
It cannot dip below it: the dashed line is the Eckart-Young floor, and the
network is asymptotically pinned to it.

In [ ]:
import matplotlib.pyplot as plt

pca_mse = np.mean((X_pca - Xc) ** 2)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(losses, linewidth=1.5, label="autoencoder training MSE")
ax.axhline(pca_mse, linestyle="--", color="0.4",
           label=f"PCA (Eckart-Young) MSE = {pca_mse:.2e}")
ax.set_yscale("log")
ax.set_xlabel("Gradient-descent epoch")
ax.set_ylabel("Mean squared reconstruction error")
ax.set_title("Linear autoencoder descends to the PCA optimum")
ax.legend(loc="upper right")
fig;

### 1c. Do the two reconstructions actually match?

A loss curve landing on the right floor is suggestive; the decisive test is
whether the two methods reconstruct the *same data* the same way. We compare the
reconstructions directly, and -- more stringently -- measure the principal angles
between the subspace the autoencoder's decoder spans and the top-$k$ PCA
subspace. Angles near zero mean the network found the principal subspace itself,
not merely a comparable error.

In [ ]:
# Direct agreement of the two reconstructions.
recon_gap = reconstruction_error(X_pca, X_ae, kind="rel_l2")

# Subspace agreement: principal angles between decoder row-space and PCA components.
Q_ae, _ = np.linalg.qr(W_d.T)   # orthonormal basis for the autoencoder subspace
Q_pca, _ = np.linalg.qr(Vt.T)   # orthonormal basis for the top-k PCA subspace
cos_angles = np.clip(np.linalg.svd(Q_ae.T @ Q_pca, compute_uv=False), -1.0, 1.0)
principal_angles_deg = np.degrees(np.arccos(cos_angles))

print(f"Autoencoder-vs-PCA reconstruction difference (rel-L2): {recon_gap:.4f}")
print(f"Principal angles between subspaces (deg): "
      f"{np.round(principal_angles_deg, 3)}")

The reconstructions agree to a fraction of a percent and the principal angles are
essentially zero. That is the equivalence made concrete: a network trained only
to reconstruct its input, with no knowledge of eigenvectors, has rediscovered the
principal subspace. A scatter of one method's reconstruction of a single feature against the other's makes the point visually -- the points sit on the identity line.

In [ ]:
feat = 0  # inspect a single feature dimension across all samples

fig, ax = plt.subplots(figsize=(5.2, 5))
lo = min(X_pca[:, feat].min(), X_ae[:, feat].min())
hi = max(X_pca[:, feat].max(), X_ae[:, feat].max())
ax.plot([lo, hi], [lo, hi], linestyle="--", color="0.5", label="identity")
ax.scatter(X_pca[:, feat], X_ae[:, feat], s=14, alpha=0.6, label="samples")
ax.set_xlabel(f"PCA reconstruction (feature {feat})")
ax.set_ylabel(f"Autoencoder reconstruction (feature {feat})")
ax.set_title("Per-sample reconstructions coincide")
ax.legend(loc="upper left")
fig;

## 2. A biomedical latent space

The synthetic fixture proved the equivalence; now we point the same machinery at
a real biomedical measurement. The Wisconsin **breast-cancer** dataset bundled
with scikit-learn describes 569 tumor samples by 30 morphological features
computed from digitized fine-needle-aspirate images (cell radius, texture,
concavity, and so on), each labeled benign or malignant. This is the kind of
wide, correlated feature matrix where a low-dimensional latent is genuinely
useful: many of the 30 features are near-redundant descriptors of a few
underlying tumor properties.

Because the features live on wildly different numeric scales, we standardize each
to zero mean and unit variance first -- otherwise PCA (and the autoencoder) would
simply chase whichever feature happens to have the largest raw units.

In [ ]:
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer()
X_raw = data.data                       # (569, 30) morphological features
y = data.target                         # 0 = malignant, 1 = benign

# Standardize features, then center for the decomposition.
X_std = (X_raw - X_raw.mean(axis=0)) / X_raw.std(axis=0)
X_bio = X_std - X_std.mean(axis=0, keepdims=True)

evr_bio = explained_variance_ratio(X_std)
print(f"Feature matrix: {X_raw.shape} (tumors x features)")
print(f"Class balance (malignant, benign): {np.bincount(y)}")
print(f"Top-2 components capture {evr_bio[:2].sum():.2%} of variance.")

We compress to a two-dimensional latent with both methods and confirm they agree
on real data too, then read the latent space. Unlike the synthetic fixture, no
two components capture *all* the variance here -- a real 30-feature tumor
description is not exactly rank 2 -- so the reconstruction error is substantial
and honest. The question is whether the two dimensions we keep are biologically
organized.

In [ ]:
k_bio = 2

# Closed-form PCA reconstruction.
U_b, s_b, Vt_b = svd_lowrank(X_bio, k_bio)
X_bio_pca = U_b @ np.diag(s_b) @ Vt_b
err_bio_pca = reconstruction_error(X_bio, X_bio_pca, kind="rel_l2")

# Linear autoencoder trained the same way as before.
d_b, N_b = X_bio.shape[1], X_bio.shape[0]
ae_rng = np.random.default_rng(2)
We_b = 0.1 * ae_rng.standard_normal((d_b, k_bio))
Wd_b = 0.1 * ae_rng.standard_normal((k_bio, d_b))
lr_b, epochs_b = 0.02, 8000
for _ in range(epochs_b):
    Zb = X_bio @ We_b
    Rb = Zb @ Wd_b - X_bio
    Wd_b -= lr_b * (Zb.T @ Rb) * (2.0 / (N_b * d_b))
    We_b -= lr_b * (X_bio.T @ (Rb @ Wd_b.T)) * (2.0 / (N_b * d_b))
X_bio_ae = (X_bio @ We_b) @ Wd_b
err_bio_ae = reconstruction_error(X_bio, X_bio_ae, kind="rel_l2")

print(f"PCA (k=2) reconstruction error:        {err_bio_pca:.4f}")
print(f"Autoencoder (k=2) reconstruction error: {err_bio_ae:.4f}")
print(f"Method-to-method difference (rel-L2):   "
      f"{reconstruction_error(X_bio_pca, X_bio_ae, kind='rel_l2'):.4f}")

The two latents reconstruct the tumors almost identically -- the equivalence is
not an artifact of the clean synthetic data. Now the biomedical payoff: we plot
each tumor by its two PCA latent scores and color by diagnosis. A linear latent
is only worth keeping if the biology organizes along it.

In [ ]:
from ddm4bio.methods.decomposition import pca_reduce

scores = pca_reduce(X_std, n_components=2)

fig, ax = plt.subplots(figsize=(7, 5.5))
for label, name in [(0, "malignant"), (1, "benign")]:
    sel = y == label
    ax.scatter(scores[sel, 0], scores[sel, 1], s=18, alpha=0.65, label=name)
ax.set_xlabel("Latent dimension 1")
ax.set_ylabel("Latent dimension 2")
ax.set_title("Two-dimensional linear latent of breast-cancer morphology")
ax.legend(title="diagnosis", loc="upper right")
fig;

## 3. Where nonlinearity earns its keep: a nonlinear autoencoder

The equivalence above is a *floor*, not a ceiling. A linear latent can only draw a
flat subspace through the data; when the real structure is a **curved manifold**,
no plane can capture it. Single-cell transcriptomes are the canonical case -- cells
along a differentiation trajectory trace a curve through gene-expression space --
but we can see the effect cleanly on a fixture whose truth we control: the
**S-curve**, a two-dimensional sheet folded into three dimensions.

We compress it to a two-dimensional latent two ways: the best linear plane (PCA),
and a small **nonlinear autoencoder** with the same encoder->latent->decoder shape
but `tanh` nonlinearities that let the latent surface bend to follow the fold.

In [ ]:
import torch.nn as nn
from sklearn.datasets import make_s_curve

# A 2-D sheet folded into 3-D: the intrinsic dimension is 2, but it is CURVED.
X_curve, position = make_s_curve(1500, noise=0.05, random_state=0)
X_curve = (X_curve - X_curve.mean(0)) / X_curve.std(0)

# Linear baseline: PCA to 2 dims is the best flat plane through the sheet.
X_centered = X_curve - X_curve.mean(0)
_, _, Vt = np.linalg.svd(X_centered, full_matrices=False)
pca_latent = X_centered @ Vt[:2].T
pca_recon = pca_latent @ Vt[:2] + X_curve.mean(0)
linear_err = np.linalg.norm(X_curve - pca_recon) / np.linalg.norm(X_curve)


class Autoencoder(nn.Module):
    '''Encoder -> latent -> decoder, with tanh nonlinearities between layers.'''

    def __init__(self, n_features, hidden, latent):
        super().__init__()
        self.encode = nn.Sequential(
            nn.Linear(n_features, hidden), nn.Tanh(), nn.Linear(hidden, latent)
        )
        self.decode = nn.Sequential(
            nn.Linear(latent, hidden), nn.Tanh(), nn.Linear(hidden, n_features)
        )

    def forward(self, x):
        return self.decode(self.encode(x))


torch.manual_seed(0)
X_curve_t = torch.tensor(X_curve, dtype=torch.float32, device=device)
autoencoder = Autoencoder(n_features=3, hidden=32, latent=2).to(device)
optimizer = torch.optim.Adam(autoencoder.parameters(), lr=1e-2)
for epoch in range(1500):
    optimizer.zero_grad()
    loss = ((autoencoder(X_curve_t) - X_curve_t) ** 2).mean()
    loss.backward()
    optimizer.step()

with torch.no_grad():
    ae_err = (
        torch.linalg.norm(autoencoder(X_curve_t) - X_curve_t)
        / torch.linalg.norm(X_curve_t)
    ).item()
    ae_latent = autoencoder.encode(X_curve_t).cpu().numpy()

print("Curved 2-D manifold in 3-D, compressed to a 2-D latent:")
print(f"  linear (PCA plane)    reconstruction rel-L2 = {linear_err:.3f}")
print(f"  nonlinear autoencoder reconstruction rel-L2 = {ae_err:.3f}")
print("  -> the nonlinear latent follows the fold; the flat plane cannot.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
for ax, latent, title in [
    (axes[0], pca_latent, f"Linear (PCA) latent -- rel-L2 {linear_err:.2f}"),
    (axes[1], ae_latent, f"Nonlinear autoencoder latent -- rel-L2 {ae_err:.2f}"),
]:
    dots = ax.scatter(latent[:, 0], latent[:, 1], c=position, cmap="viridis", s=8)
    ax.set_xlabel("latent dimension 1")
    ax.set_ylabel("latent dimension 2")
    ax.set_title(title)
fig.colorbar(dots, ax=axes, label="position along the manifold", shrink=0.8)
fig;

The colour is the true position along the fold. The linear latent smears
those positions together -- a flat plane cannot separate points that the fold
brings close in 3-D -- while the nonlinear autoencoder lays them out as a smooth
gradient: it has *unrolled* the manifold, halving the reconstruction error at the
same latent dimension. That is the whole reason to reach for a nonlinear model:
not more parameters for their own sake, but a latent that can bend to the data.

## 4. SHRED: reconstructing a whole field from a few sensors

Week 7 asked *how far ahead* you can forecast a dynamical system. Here is the
complementary question: *how few measurements* do you need to see the whole thing?
**SHRED** -- a SHallow REcurrent DEcoder -- answers it. Rather than measure a
spatiotemporal field everywhere, you place a handful of **sensors** at fixed
locations, feed their short recent history to a small recurrent network (an LSTM),
and let a decoder reconstruct the *entire* spatial field at each moment.

We build a field of a few travelling waves over space and time, hide all but a few
sensor locations, and ask SHRED to recover the full field on held-out snapshots it
never trained on. The striking result -- from Kutz's work -- is how few sensors it
takes.

In [ ]:
# A spatiotemporal field: a few travelling waves over space (columns), time (rows).
n_time, n_space = 500, 48
space = np.linspace(0, 2 * np.pi, n_space)
clock = np.linspace(0, 30, n_time)
field = (
    np.sin(space[None] - 1.3 * clock[:, None])
    + 0.6 * np.sin(2 * space[None] + 0.7 * clock[:, None])
    + 0.4 * np.cos(3 * space[None] - 0.4 * clock[:, None])
)
field = (field - field.mean()) / field.std()


class Shred(nn.Module):
    '''Shallow recurrent decoder: an LSTM over sensor history, then a field decoder.'''

    def __init__(self, n_sensors, hidden, n_space):
        super().__init__()
        self.lstm = nn.LSTM(n_sensors, hidden, batch_first=True)
        self.decode = nn.Sequential(nn.Linear(hidden, 96), nn.ReLU(), nn.Linear(96, n_space))

    def forward(self, x):
        history, _ = self.lstm(x)
        return self.decode(history[:, -1])


def train_shred(n_sensors, lags=20, hidden=40, epochs=350, seed=0):
    '''Place `n_sensors` random sensors; train SHRED to reconstruct the full field.'''
    rng = np.random.default_rng(seed)
    torch.manual_seed(seed)
    sensors = np.sort(rng.choice(n_space, n_sensors, replace=False))
    readings = field[:, sensors]
    # Each sample: `lags` recent sensor readings -> the full field at that instant.
    sequences = np.stack([readings[i - lags:i] for i in range(lags, n_time)])
    targets = field[lags:]
    perm = rng.permutation(len(sequences))
    n_train = int(0.75 * len(sequences))
    train_idx, test_idx = perm[:n_train], perm[n_train:]
    as_tensor = lambda a: torch.tensor(a, dtype=torch.float32, device=device)

    model = Shred(n_sensors, hidden, n_space).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=4e-3)
    x_train, y_train = as_tensor(sequences[train_idx]), as_tensor(targets[train_idx])
    x_test, y_test = as_tensor(sequences[test_idx]), as_tensor(targets[test_idx])
    for epoch in range(epochs):
        optimizer.zero_grad()
        loss = ((model(x_train) - y_train) ** 2).mean()
        loss.backward()
        optimizer.step()
    with torch.no_grad():
        err = (torch.linalg.norm(model(x_test) - y_test) / torch.linalg.norm(y_test)).item()
        full_reconstruction = model(as_tensor(sequences)).cpu().numpy()
    return err, sensors, full_reconstruction


sensor_errors = {}
for k in (1, 2, 3, 5):
    err, sensors, reconstruction = train_shred(k)
    sensor_errors[k] = err
    if k == 3:
        sensors_3, reconstruction_3 = sensors, reconstruction
    print(f"  {k} sensor(s) of {n_space}: held-out field reconstruction rel-L2 = {err:.3f}")
print(f"  -> just 3 sensors of {n_space} reconstruct the full field to "
      f"{sensor_errors[3]:.1%} error.")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
vmin, vmax = field.min(), field.max()
axes[0].imshow(field[20:].T, aspect="auto", cmap="RdBu_r", vmin=vmin, vmax=vmax)
axes[0].set_title("True field (3 sensor rows marked)")
for row in sensors_3:
    axes[0].axhline(row, color="k", lw=0.8, ls=":")
axes[1].imshow(reconstruction_3.T, aspect="auto", cmap="RdBu_r", vmin=vmin, vmax=vmax)
axes[1].set_title(f"SHRED from 3 sensors -- rel-L2 {sensor_errors[3]:.2f}")
for ax in axes[:2]:
    ax.set_xlabel("time")
    ax.set_ylabel("space")
axes[2].plot(list(sensor_errors), list(sensor_errors.values()), "o-")
axes[2].set_xlabel("number of sensors")
axes[2].set_ylabel("held-out rel-L2 error")
axes[2].set_title("A few sensors suffice")
fig.tight_layout()
fig;

Three sensors -- a tiny fraction of the domain -- reconstruct the entire
field, and the error falls sharply as the first few sensors are added before it
plateaus. The recurrent network learns to read the field's low-dimensional
dynamics off the sensor histories, exactly the kind of structure Weeks 5-7 taught
us to look for; SHRED just decodes it back out to full resolution.

## 5. Interpretation

In [ ]:
from ddm4bio.interpret import interpretation_block

block = interpretation_block(
    claim=(
        f"A linear autoencoder is exactly PCA (subspaces agreed to "
        f"{principal_angles_deg.max():.2f} degrees); a nonlinear autoencoder beats "
        f"the flat linear latent on a curved manifold ({linear_err:.2f} -> "
        f"{ae_err:.2f} relative-L2 at the same latent dimension); and SHRED "
        f"reconstructs a full spatiotemporal field from just 3 of {n_space} sensors "
        f"({sensor_errors[3]:.1%} error)."
    ),
    confidence="high",
    limitations_list=[
        "The linear=PCA equivalence has a closed-form guarantee (Eckart-Young); the "
        "nonlinear-autoencoder and SHRED results do NOT -- they are empirical and "
        "depend on initialization and training, and must be reported with that caveat.",
        "The fixtures are small and seeded (an S-curve and a low-rank wave field) so "
        "the lesson runs in seconds on a CPU; real single-cell manifolds and turbulent "
        "fields are far higher-dimensional and noisier.",
        "SHRED is scored on held-out snapshots drawn at random (interpolation); "
        "reconstructing the future (extrapolation) is a harder, separate problem.",
        "The genuinely large models -- transformers, foundation-model embeddings, and "
        "reinforcement learning -- are the subject of the capstone, not this lesson.",
    ],
    evidence=(
        "an exact linear-autoencoder/PCA subspace match, a nonlinear autoencoder that "
        "halves the reconstruction error of the best linear plane on a curved manifold, "
        "and a SHRED reconstruction of a full field from three sensors"
    ),
)
print(block)

## Exercises

Week 8 has no separate problem set. Your graded work is the **capstone project**,
where you carry a dataset of your choice through the full ddm4bio pipeline --
honest QC, a validated method, an interpretation block -- end to end, and where the
genuinely large deep-learning models are taught through a real case study. Building
on this lesson, the capstone takes on:

- **Nonlinear representation learning** -- choose a representation for your data and
  justify it against the linear baseline: show what a PCA / linear-autoencoder
  latent captures, and argue whether your data need a nonlinear autoencoder or
  whether the linear latent already suffices.
- **Reconstruction from sparse measurement** -- where your problem has a
  spatiotemporal or multi-sensor structure, use SHRED-style reconstruction and
  report how few sensors you can afford.
- **Foundation models and sequences** -- use pretrained transformer /
  foundation-model *embeddings* for a clinical-text or biological-sequence task
  (using a pretrained model, not training one from scratch).
- **Reinforcement learning** -- for a sequential-decision problem such as dosing,
  frame a small policy-learning task and report its behaviour honestly against a
  simple baseline.
- Close with an `interpretation_block` stating a calibrated confidence and named
  limitations, with special attention to what your chosen model can and cannot
  represent.

Refer to the capstone project brief for scope, deliverables, and grading.